# Challenge — Red neuronal con Keras 3
### Universidad EAFIT | SI3003 — Introducción a la Inteligencia Artificial

---

## Objetivo

En clase construimos una red neuronal de clasificación con **Keras 3**.

En este ejercicio vas a aplicar la misma receta sobre un dataset diferente:

> **Olivetti Faces**

El dataset contiene imágenes de rostros en escala de grises. El objetivo será identificar a cuál persona pertenece cada imagen.

No necesitas diseñar un pipeline de datos nuevo. La carga, división y visualización están resueltas.

Tu trabajo se concentra en completar las etapas principales de Keras:

```text
Datos
  ↓
Modelo
  ↓
Loss + Optimizer
  ↓
Entrenamiento
  ↓
Evaluación
  ↓
Predicción
```

---

## ¿Qué debes completar?

Las celdas marcadas con `# TODO` contienen espacios que debes completar.

El resto del código está preparado para que puedas concentrarte en los conceptos vistos en clase.


---
## 0. Importar librerías

Usaremos:

- **Keras 3** para construir y entrenar la red.
- **scikit-learn** para cargar Olivetti Faces y dividir los datos.
- **NumPy** para manipular arrays.
- **Matplotlib** para visualizar imágenes.


In [ ]:
import keras
from keras import layers

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_olivetti_faces
from sklearn.model_selection import train_test_split

SEED = 42
keras.utils.set_random_seed(SEED)

print(f"Keras version : {keras.__version__}")
print(f"Backend       : {keras.backend.backend()}")


---
# 1. Dataset: Olivetti Faces

Olivetti Faces contiene:

- **400 imágenes**
- imágenes de **64 × 64 píxeles**
- escala de grises
- **40 personas**
- **10 imágenes por persona**

La etiqueta indica la identidad de la persona:

```text
0, 1, 2, ..., 39
```

Cada imagen ya viene representada con valores flotantes aproximadamente en el rango:

$$
[0,1]
$$

Por eso, en este ejercicio **no necesitamos normalizar dividiendo por 255**.


In [ ]:
# Esta parte está resuelta.

faces = fetch_olivetti_faces(
    shuffle=True,
    random_state=SEED
)

X = faces.images
y = faces.target

print("X:", X.shape)
print("y:", y.shape)
print("Valor mínimo:", X.min())
print("Valor máximo:", X.max())
print("Número de clases:", len(np.unique(y)))


## 1.1 Dividir entrenamiento y prueba

Como solo tenemos 10 imágenes por persona, debemos asegurarnos de que todas las identidades aparezcan tanto en entrenamiento como en prueba.

Usaremos una división **estratificada**:

```text
80% entrenamiento
20% prueba
```

La opción:

```python
stratify=y
```

mantiene la proporción de cada clase en ambos conjuntos.

Esta parte está resuelta.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nPersonas en train:", len(np.unique(y_train)))
print("Personas en test :", len(np.unique(y_test)))


### Pregunta 1

Cada imagen tiene tamaño:

$$
64 \times 64
$$

Cuando apliquemos `Flatten()`, ¿cuántos valores tendrá cada ejemplo?

**Respuesta:** Cada imagen tendrá **4096 valores** (64 × 64 = 4096). `Flatten()` solo reorganiza la matriz 2D en un vector 1D; no cambia ni agrega información, solo la forma. Por eso la primera capa `Dense(64)` tiene 4096·64 + 64 = **262,208** parámetros.


---
## 1.2 Visualizar algunas imágenes

Esta parte está resuelta.

Observa que varias fotografías corresponden a la misma persona con pequeñas variaciones de expresión, iluminación y orientación.


In [ ]:
plt.figure(figsize=(12, 6))

for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_train[i], cmap="gray")
    plt.title(f"Persona {y_train[i]}")
    plt.xticks([])
    plt.yticks([])

plt.tight_layout()
plt.show()


---
# 2. Construir la red neuronal

Usaremos una arquitectura fully-connected similar a la vista en clase:

```text
Imagen 64×64
     ↓
Flatten
     ↓
4096 valores
     ↓
Dense(64)
     ↓
ReLU
     ↓
Dense(40)
     ↓
Logits
```

La última capa debe tener una salida por cada clase.

> **Importante:** no añadas `Softmax` al final. Trabajaremos directamente con **logits**.


In [ ]:
# TODO 1 — Completar la arquitectura.

model = keras.Sequential([

    keras.Input(shape=(64, 64)),

    layers.Flatten(),

    layers.Dense(
        64,
        activation="relu"
    ),

    layers.Dense(40)
])

model.summary()


In [ ]:
# Validación TODO 1

assert isinstance(model, keras.Model)

# Dense(64): 4096*64 + 64
# Dense(40): 64*40 + 40
expected_params = (4096 * 64 + 64) + (64 * 40 + 40)

assert model.count_params() == expected_params, (
    f"Se esperaban {expected_params:,} parámetros, "
    f"pero el modelo tiene {model.count_params():,}."
)

print("✓ Arquitectura correcta.")
print(f"Parámetros entrenables: {model.count_params():,}")


### Pregunta 2

¿Por qué necesitamos `Flatten()` antes de la primera capa `Dense`?

**Respuesta:** Porque `Dense` espera, por cada ejemplo, un **vector** de entrada, y cada imagen es una matriz de 64×64. `Flatten()` la convierte en un vector de 4096 valores para que cada neurona de la capa densa pueda conectarse con todos los píxeles. Sin ese paso, la forma de entrada no sería la que `Dense` necesita.





### Pregunta 3

¿Por qué la última capa tiene **40 neuronas**?

**Respuesta:** Porque el problema tiene **40 clases** (40 personas distintas, etiquetadas de 0 a 39). La última capa produce un **logit por clase**, es decir, una puntuación para cada persona; la clase predicha es la de mayor logit. Si tuviera otro número de neuronas, la salida no se correspondería con las etiquetas.





### Pregunta 4

¿Qué función cumple `ReLU` en la capa oculta?

**Respuesta:** Introduce **no linealidad** (`ReLU(x) = max(0, x)`). Sin ella, apilar capas `Dense` equivaldría a una sola transformación lineal y la red no podría aprender fronteras de decisión complejas entre rostros. Además es barata de calcular y ayuda a mitigar el problema del gradiente que se desvanece.




---
# 3. Configurar el entrenamiento

Las etiquetas son números enteros entre:

```text
0 y 39
```

Por eso usaremos:

```python
SparseCategoricalCrossentropy
```

Además:

- el modelo produce **logits**,
- usaremos **Adam**,
- mediremos **accuracy**.

Completa `model.compile()`.


In [ ]:
lr_rate = 0.001

# TODO 2 — Configurar loss, optimizer y métrica.

model.compile(

    optimizer=keras.optimizers.Adam(
        learning_rate=lr_rate
    ),

    loss=keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),

    metrics=["accuracy"]
)

print("✓ Modelo configurado.")


### Pregunta 5

¿Por qué usamos:

```python
from_logits=True
```

en la función de pérdida?

**Respuesta:** Porque la última capa `Dense(40)` **no tiene Softmax**: entrega logits (números reales sin normalizar). `from_logits=True` le indica a la función de pérdida que debe aplicar internamente la softmax antes de calcular la entropía cruzada. Hacerlo así es **numéricamente más estable** que aplicar Softmax en el modelo y luego tomar el logaritmo (se usa la forma log-sum-exp). Si usáramos `from_logits=False` con logits, la pérdida se calcularía mal, porque esperaría probabilidades.


---
# 4. Entrenar el modelo

Usaremos:

```python
epochs = 30
batch_size = 32
```

El dataset es pequeño, por lo que el entrenamiento debería ser rápido.

Keras realizará internamente:

```text
Forward pass
      ↓
Calcular loss
      ↓
Backpropagation
      ↓
Actualizar parámetros con Adam
```

Completa la llamada a `fit()`.


In [ ]:
epochs = 30
batch_size = 32

# TODO 3 — Entrenar el modelo.

history = model.fit(

    X_train,
    y_train,

    epochs=epochs,
    batch_size=batch_size,

    validation_data=(X_test, y_test),

    shuffle=True
)


### Pregunta 6

¿Qué representa una **epoch**?

**Respuesta:** Una pasada completa por **todo** el conjunto de entrenamiento (aquí, las 320 imágenes). Con `epochs=30`, la red ve cada imagen de entrenamiento 30 veces.





### Pregunta 7

¿Qué significa `batch_size=32`?

**Respuesta:** Que los pesos se actualizan después de procesar **32 ejemplos** (un mini-batch). Con 320 imágenes hay 320/32 = **10 actualizaciones por época**, es decir, 300 en las 30 épocas.


---
## 4.1 Visualizar el entrenamiento

Esta parte está resuelta.

El dataset tiene muy pocos ejemplos comparado con el número de parámetros de la red.

Observa cuidadosamente la diferencia entre entrenamiento y validación.


In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(
    history.history["accuracy"],
    label="Train accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Evolución del entrenamiento")
plt.legend()

plt.show()


### Pregunta 8

Observando la gráfica:

- ¿El accuracy de entrenamiento sigue aumentando?
- ¿El accuracy de validación se comporta igual?
- ¿Observas evidencia de **overfitting**?

**Respuesta:** Se evalúa con dos criterios: (a) la **brecha** entre accuracy de entrenamiento y de validación, y (b) el comportamiento de la `val_loss` frente a la `train_loss`. Lo esperable con esta red es que el accuracy de **entrenamiento siga subiendo** hasta acercarse a 1.0 (la red puede memorizar 320 imágenes), mientras que el de **validación sube más lento y se estanca** por debajo, dejando una brecha entre ambas curvas. Esa brecha es la evidencia de **overfitting**. La celda siguiente aplica estos criterios con los números reales de tu ejecución.


In [ ]:
# Pregunta 8 — diagnóstico de overfitting con los números reales
from IPython.display import Markdown, display

h = history.history
train_acc, val_acc = h["accuracy"][-1], h["val_accuracy"][-1]
gap = train_acc - val_acc
best_ep = int(np.argmin(h["val_loss"])) + 1
val_loss_rises = h["val_loss"][-1] > min(h["val_loss"]) + 1e-4
GAP_THRESHOLD = 0.05   # SUPUESTO: brecha train-val > 5 puntos = señal de overfitting

hay = gap > GAP_THRESHOLD or (val_loss_rises and best_ep < len(h["val_loss"]))
display(Markdown(
    f"- Accuracy final: train = **{train_acc:.3f}**, validación = **{val_acc:.3f}** (brecha = {gap:+.3f}).\n"
    f"- Menor `val_loss` en la época **{best_ep}** de {len(h['val_loss'])}; "
    f"`val_loss` final = {h['val_loss'][-1]:.3f} ({'subió' if val_loss_rises else 'no subió'} respecto a su mínimo).\n"
    f"- **Conclusión:** {'sí hay evidencia de overfitting' if hay else 'no hay evidencia clara de overfitting'} "
    f"(umbral de brecha = {GAP_THRESHOLD})."
))


---
# 5. Evaluar el modelo

Ahora evaluaremos el modelo sobre imágenes que no fueron utilizadas para actualizar sus parámetros.

Completa `evaluate()`.


In [ ]:
# TODO 4 — Evaluar.

test_loss, test_accuracy = model.evaluate(

    X_test,
    y_test,

    batch_size=batch_size,
    verbose=0
)

print(f"Test loss     : {test_loss:.4f}")
print(f"Test accuracy : {test_accuracy:.4f}")
print(f"Accuracy (%)  : {test_accuracy * 100:.2f}%")


### Pregunta 9

Tenemos **40 clases**, pero solamente **320 imágenes de entrenamiento**.

¿Crees que la cantidad de datos es grande o pequeña para una red con cientos de miles de parámetros?

**Respuesta:** **Pequeña.** La red tiene 4096·64 + 64 + 64·40 + 40 = **264,808 parámetros** para solo 320 imágenes de entrenamiento: unos 800 parámetros por ejemplo y apenas 8 imágenes por persona. Con tantos grados de libertad, la red puede memorizar los ejemplos de entrenamiento (incluidos detalles irrelevantes como la iluminación) en lugar de aprender lo que identifica a cada rostro, lo que la hace propensa al overfitting. Aun así, Olivetti es un dataset fácil (fondo oscuro, rostros frontales y centrados), por lo que el accuracy de prueba puede ser razonable; la limitación es la **fiabilidad de la generalización**, no que sea imposible aprender.


In [ ]:
# Pregunta 9 — cuántos parámetros por ejemplo de entrenamiento
n_params = model.count_params()
n_train = len(X_train)
n_por_clase = n_train // len(np.unique(y_train))

from IPython.display import Markdown, display
display(Markdown(
    f"- Parámetros de la red: **{n_params:,}**\n"
    f"- Imágenes de entrenamiento: **{n_train}** ({n_por_clase} por persona)\n"
    f"- Parámetros por imagen de entrenamiento: **{n_params / n_train:,.0f}**"
))


---
# 6. Logits y Softmax

La salida de nuestro modelo tiene forma:

```text
(batch_size, 40)
```

Cada uno de los 40 valores corresponde al **logit** asociado a una persona.

Los logits no son probabilidades.

Para convertirlos en probabilidades usamos:

```python
keras.ops.softmax(...)
```


In [ ]:
index = 0

image = X_test[index]
true_class = y_test[index]

plt.imshow(image, cmap="gray")
plt.title(f"Persona real: {true_class}")
plt.axis("off")
plt.show()

# Añadimos dimensión de batch:
# (64,64) -> (1,64,64)
image_batch = np.expand_dims(image, axis=0)

# Forward pass.
logits = model(
    image_batch,
    training=False
)

print("Shape de logits:", logits.shape)
print("\nPrimeros 10 logits:")
print(
    keras.ops.convert_to_numpy(logits)[0, :10]
)


In [ ]:
# TODO 5 — Convertir logits a probabilidades.

probabilities = keras.ops.softmax(logits)

# TODO 6 — Obtener la clase con mayor probabilidad.

prediction = keras.ops.argmax(
    probabilities,
    axis=1
)

prediction = int(
    keras.ops.convert_to_numpy(prediction)[0]
)

print()
print("Persona real     :", true_class)
print("Persona predicha :", prediction)


In [ ]:
# Revisamos las cinco clases con mayor probabilidad.

probabilities_np = keras.ops.convert_to_numpy(
    probabilities
)[0]

top5 = np.argsort(
    probabilities_np
)[-5:][::-1]

print("Top 5 predicciones:\n")

for class_id in top5:
    print(
        f"Persona {class_id:2d}: "
        f"{probabilities_np[class_id]:.4f}"
    )

print(
    "\nSuma de probabilidades:",
    probabilities_np.sum()
)


### Pregunta 10

¿Por qué la suma de las 40 probabilidades obtenidas con `Softmax` debe ser aproximadamente `1`?

**Respuesta:** Porque Softmax convierte los logits en una **distribución de probabilidad** sobre las 40 personas: elevamos cada logit con `exp` (para hacerlo positivo) y lo dividimos por la suma de todos, `softmax(z_i) = exp(z_i) / Σ_j exp(z_j)`. Al dividir cada término por la suma total, la suma de las 40 probabilidades es exactamente 1 por construcción (pequeñas diferencias, como 0.99999994, son error de punto flotante).


---
# 7. Visualizar varias predicciones

Esta parte está resuelta.

Observa especialmente los casos en los que el modelo se equivoca.


In [ ]:
n = 12

logits_batch = model(
    X_test[:n],
    training=False
)

predictions = keras.ops.argmax(
    logits_batch,
    axis=1
)

predictions = keras.ops.convert_to_numpy(
    predictions
)

plt.figure(figsize=(12, 8))

for i in range(n):

    plt.subplot(3, 4, i + 1)

    plt.imshow(
        X_test[i],
        cmap="gray"
    )

    real = y_test[i]
    pred = predictions[i]

    plt.title(
        f"Real: {real}\nPred: {pred}",
        fontsize=9
    )

    plt.xticks([])
    plt.yticks([])

plt.tight_layout()
plt.show()


---
# 8. Reflexión final

Responde brevemente.

### 1. ¿Cuál fue el accuracy final del modelo?

**Respuesta:** Ver la celda de código siguiente: imprime el accuracy de test de **tu ejecución** (junto con el de entrenamiento y el número de errores sobre las 80 imágenes de prueba). Como referencia, el azar con 40 clases sería 1/40 = 2.5%.


### 2. ¿Observaste overfitting? ¿Qué evidencia utilizaste?

**Respuesta:** Se determina con los mismos dos criterios de la Pregunta 8: (a) la **brecha** entre train accuracy y validation accuracy al final del entrenamiento, y (b) si la `val_loss` se estanca (o sube) mientras la `train_loss` sigue bajando. La celda siguiente calcula ambos con los números reales de tu ejecución y emite el veredicto.


### 3. ¿Qué ocurre cuando tenemos muchos parámetros pero pocos ejemplos de entrenamiento?

**Respuesta:** La red tiende a **memorizar** el conjunto de entrenamiento en lugar de aprender patrones generales: el error de entrenamiento baja casi a cero, pero el de validación/prueba queda más alto (alta varianza, mala generalización). Con 264,808 parámetros y 320 ejemplos, la red tiene capacidad de sobra para ajustar hasta el ruido. Se mitiga con más datos, *data augmentation*, regularización (`Dropout`, L2/weight decay), una red más pequeña, *early stopping* o arquitecturas con menos parámetros efectivos (CNN).


### 4. Después de `Flatten()`, una cara de 64×64 se convierte en un vector de 4096 números.

¿Qué información sobre la estructura de la imagen deja de representar explícitamente esta transformación?

**Respuesta:** Deja de representar **explícitamente la estructura espacial 2D**: qué píxeles son vecinos (arriba/abajo/izquierda/derecha), los bordes, las formas y los patrones locales (ojos, nariz, boca). Para `Dense`, el píxel de al lado y el de la esquina opuesta son simplemente dos entradas más, cada una con su propio peso. De hecho, si se permutaran los píxeles de todas las imágenes con la misma permutación aleatoria, la red densa aprendería igual de bien, porque no usa la posición; un humano, en cambio, ya no reconocería nada. Tampoco hay invarianza a la traslación: si el rostro se desplaza unos píxeles, el vector de entrada cambia por completo.


### 5. ¿Crees que una arquitectura diseñada específicamente para imágenes podría aprovechar mejor la información espacial?

**Respuesta:** **Sí.** Una **red convolucional (CNN)** usa filtros locales que comparten pesos y recorren toda la imagen, así que detecta bordes, texturas y partes del rostro en cualquier posición, y las capas apiladas construyen jerarquías (bordes → partes del rostro → rostro). Además usa muchos menos parámetros que una capa densa sobre 4096 entradas, lo cual es especialmente útil con solo 320 imágenes de entrenamiento, porque reduce el riesgo de overfitting.


---

## Para pensar

Nuestra red recibe:

```text
64 × 64
   ↓
Flatten
   ↓
4096 números
   ↓
Dense
```

Pero una imagen tiene una estructura espacial:

- píxeles cercanos están relacionados,
- existen bordes,
- existen formas,
- existen patrones locales.

Una red fully-connected no incorpora explícitamente esa estructura.

Esto motiva una arquitectura diseñada específicamente para imágenes:

> **Convolutional Neural Networks (CNNs)**


In [ ]:
# Reflexión 1 y 2 — resultados reales de esta ejecución
from IPython.display import Markdown, display

h = history.history
train_acc, val_acc = h["accuracy"][-1], h["val_accuracy"][-1]
gap = train_acc - val_acc

y_pred = keras.ops.convert_to_numpy(
    keras.ops.argmax(model.predict(X_test, batch_size=batch_size, verbose=0), axis=1)
)
errores = np.where(y_pred != y_test)[0]

txt  = f"**1. Accuracy final (test):** {test_accuracy:.4f} ({test_accuracy*100:.2f}%); el azar sería 1/40 = 2.50%.\n\n"
txt += f"- Accuracy final de entrenamiento: {train_acc:.4f}\n"
txt += f"- Errores en test: **{len(errores)}** de {len(y_test)} imágenes.\n\n"
if len(errores):
    pares = "; ".join(f"real {int(y_test[i])} → pred {int(y_pred[i])}" for i in errores[:8])
    txt += f"- Primeros errores: {pares}\n\n"
txt += (f"**2. Overfitting:** brecha train − val = **{gap:+.3f}** "
        f"({'hay brecha apreciable' if gap > 0.05 else 'brecha pequeña'} con el umbral de 0.05); "
        f"`train_loss` final = {h['loss'][-1]:.4f}, `val_loss` final = {h['val_loss'][-1]:.4f}.")
display(Markdown(txt))


---
# Resumen

En este ejercicio aplicaste nuevamente la receta básica de entrenamiento de una red neuronal:

```text
Dataset
   ↓
Train / Test split
   ↓
Sequential
   ↓
Flatten
   ↓
Dense + ReLU
   ↓
Dense(40)
   ↓
Cross-Entropy
   ↓
Adam
   ↓
fit()
   ↓
evaluate()
```

Los componentes de Keras son los mismos que usamos en clase.

Lo que cambió fue:

- el dataset,
- el tamaño de las imágenes,
- el número de clases,
- y la dificultad del problema.
